# VITRA Inference-Only Ablation

This notebook runs low-cost component ablations on the EPIC clean test split without retraining. It keeps the same checkpoint and sampler split, then changes only inference inputs such as state and FOV.

## Plan

1. Run smoke ablations on a small clean EPIC subset.
2. Inspect which ablations produce a visible loss gap.
3. Run full EPIC test-split eval only for the useful settings.

The clean split parameters match the previous main eval: `CUTOFF=128000`, `seen_sampler_steps=128000`, and `exclude_seen_ids=True`.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import textwrap
import pandas as pd

RESOURCE_ROOT = Path("/kaggle/input/notebooks/ldthanh/prepare-vitra-resources")
RESOURCE_REPO = RESOURCE_ROOT / "VITRA"
RESOURCE_WHEELS = RESOURCE_ROOT / "wheels"
WORK_REPO = Path("/kaggle/working/VITRA")

if Path("/kaggle/working").exists():
    if not WORK_REPO.exists():
        assert RESOURCE_REPO.exists(), f"Missing prepared VITRA repo: {RESOURCE_REPO}. Attach prepare-vitra-resources as a Kaggle input."
        shutil.copytree(RESOURCE_REPO, WORK_REPO)
    os.chdir(WORK_REPO)
    repo = WORK_REPO
else:
    repo = Path.cwd()
    assert (repo / "scripts" / "evaluate_pretrained_loss.py").exists(), "Run locally from the repo root."

print("Repo:", repo)
print("Resource root exists:", RESOURCE_ROOT.exists())
print("Offline wheels exists:", RESOURCE_WHEELS.exists())


In [ ]:
%%bash
set -e
if [ -d /kaggle/input/notebooks/ldthanh/prepare-vitra-resources/wheels ]; then
  uv pip install scipy     "torch"     "torchvision"     "torchaudio"     "PyYAML==6.0.2"     "hydra-colorlog>=1.2.0"     "hydra-core>=1.1.1"     "deepspeed==0.16.5"     "tensorboard>=2.13.0"     "tensorboardX>=2.6.2"     "tqdm>=4.65.0"     "transformers==4.47.1"     "diffusers>=0.31.0"     "wandb>=0.19.0"     "numpy<2.0"     "sentence-transformers==2.2.2"     "open_clip_torch==2.20.0"     "datasets==2.12.0"     "draccus==0.8.0"     "einops"     "huggingface_hub"     "json-numpy"     "jsonlines"     "matplotlib"     "peft==0.11.1"     "protobuf"     "rich"     "sentencepiece==0.1.99"     "timm==0.9.10"     "tokenizers>=0.21"     "decord"     "scikit-image"     "brotli"     "imageio-ffmpeg"     "imageio"     "ffmpeg-python"     "opencv-python"     "pympler"     "ultralytics"     "pytorch-lightning"     "yacs"     "utils3d"     --no-index --find-links "/kaggle/input/notebooks/ldthanh/prepare-vitra-resources/wheels"
else
  echo "[INFO] Offline wheels not found; assuming dependencies are already installed."
fi


In [ ]:
from pathlib import Path

EVAL_SCRIPT = r

## Dataset Links

This mirrors the old pretraining notebook: create the expected `data/VITRA_1M` symlinks after copying `VITRA` from `prepare-vitra-resources`.


In [ ]:
%%bash
set -e
mkdir -p data/VITRA_1M/Video data/VITRA_1M/Annotation

# Link original EPIC videos. This mirrors SKIP_PREPROCESSING=True in the old notebook.
mkdir -p data/VITRA_1M/Video/Epic-Kitchen_root
if compgen -G "/kaggle/input/datasets/ldthanh/epic-kitchens-100-p*/*/*/*.MP4" > /dev/null; then
  for vid in /kaggle/input/datasets/ldthanh/epic-kitchens-100-p*/*/*/*.MP4; do
    ln -sf "$vid" "data/VITRA_1M/Video/Epic-Kitchen_root/$(basename "$vid")"
  done
else
  echo "[WARN] EPIC video input pattern not found. Check attached Kaggle datasets."
fi

if [ -d /kaggle/input/datasets/nahidsiddique/something-something-v2/20bn-something-something-v2/20bn-something-something-v2 ]; then
  ln -sfn /kaggle/input/datasets/nahidsiddique/something-something-v2/20bn-something-something-v2/20bn-something-something-v2 data/VITRA_1M/Video/Somethingsomething-v2_root
else
  echo "[WARN] SSV2 video root not found. EPIC-only eval can still run."
fi
if [ -d /kaggle/input/datasets/ldthanh/vitra-1m/epic/epic ]; then
  ln -sfn /kaggle/input/datasets/ldthanh/vitra-1m/epic/epic data/VITRA_1M/Annotation/epic
fi
if [ -d /kaggle/input/datasets/ldthanh/vitra-1m/ssv2/ssv2 ]; then
  ln -sfn /kaggle/input/datasets/ldthanh/vitra-1m/ssv2/ssv2 data/VITRA_1M/Annotation/ssv2
fi
if [ -d /kaggle/input/datasets/ldthanh/vitra-1m/statistics ]; then
  ln -sfn /kaggle/input/datasets/ldthanh/vitra-1m/statistics data/VITRA_1M/Annotation/statistics
fi

find data/VITRA_1M -maxdepth 3 -type l -print | head -100


In [ ]:
CONFIG = "vitra/configs/human_pretrain.json"
WEIGHTS_16K = "/kaggle/input/models/ldthanh/vitra-vla-3b/transformers/magic_mix_epic_ssv2/8/final-epoch=0-step=16000.ckpt/weights.pt"

CUTOFF = 128000
SMOKE_BATCHES = 200      # 12,800 examples with batch_size=64
FULL_BATCHES = 16400     # 1,049,600 examples, about 3h+ per setting in prior runs
NUM_WORKERS = 32

OUT_ROOT = Path("ablation_results")
OUT_ROOT.mkdir(exist_ok=True)

assert Path(CONFIG).exists(), CONFIG
assert Path(WEIGHTS_16K).exists(), WEIGHTS_16K

In [ ]:
def run_eval(name, batches, ablate_state="none", ablate_fov="none", repeated_diffusion_steps=None, phase="smoke"):
    out_dir = OUT_ROOT / phase
    out_dir.mkdir(parents=True, exist_ok=True)
    output_jsonl = out_dir / f"{name}.jsonl"
    cmd = [
        "python", "scripts/evaluate_pretrained_loss.py",
        "--config", CONFIG,
        "--weights", WEIGHTS_16K,
        "--eval_dataset", "epic",
        "--eval_sampler_step", str(CUTOFF),
        "--eval_batches", str(batches),
        "--seen_sampler_steps", str(CUTOFF),
        "--output_jsonl", str(output_jsonl),
        "--num_workers", str(NUM_WORKERS),
        "--ablate_state", ablate_state,
        "--ablate_fov", ablate_fov,
    ]
    if repeated_diffusion_steps is not None:
        cmd += ["--repeated_diffusion_steps", str(repeated_diffusion_steps)]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)
    return output_jsonl.with_name(f"{output_jsonl.stem}.epic.summary.json")

def load_summaries(phase):
    rows = []
    for path in sorted((OUT_ROOT / phase).glob("*.summary.json")):
        data = json.loads(path.read_text())
        metrics = data["metrics"]
        rows.append({
            "file": path.name,
            "checkpoint": "16k",
            "dataset": data["dataset"],
            "examples": data["eval_examples"],
            "ablate_state": data.get("ablation", {}).get("ablate_state", "none"),
            "ablate_fov": data.get("ablation", {}).get("ablate_fov", "none"),
            "repeated_diffusion_steps": data.get("ablation", {}).get("repeated_diffusion_steps"),
            "loss": metrics["loss"]["mean"],
            "left_hand_6d": metrics["left_hand_6d"]["mean"],
            "right_hand_6d": metrics["right_hand_6d"]["mean"],
            "left_hand_joints": metrics["left_hand_joints"]["mean"],
            "right_hand_joints": metrics["right_hand_joints"]["mean"],
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["examples", "ablate_state", "ablate_fov", "repeated_diffusion_steps"])
        baseline = df[(df["ablate_state"] == "none") & (df["ablate_fov"] == "none")]
        if not baseline.empty:
            base_loss = baseline.iloc[0]["loss"]
            df["delta_vs_baseline"] = df["loss"] - base_loss
            df["relative_vs_baseline_pct"] = 100 * df["delta_vs_baseline"] / base_loss
    return df


## Smoke Ablation

Run all cheap settings on the same clean EPIC subset. These are the settings to include in the report if full eval time is limited.

In [ ]:
SMOKE_SETTINGS = [
    dict(name="baseline", ablate_state="none", ablate_fov="none"),
    dict(name="zero_state", ablate_state="zero_state", ablate_fov="none"),
    dict(name="no_state", ablate_state="no_state", ablate_fov="none"),
    dict(name="shuffle_state", ablate_state="shuffle_state", ablate_fov="none"),
    dict(name="zero_fov", ablate_state="none", ablate_fov="zero_fov"),
    dict(name="shuffle_fov", ablate_state="none", ablate_fov="shuffle_fov"),
    dict(name="rds1", ablate_state="none", ablate_fov="none", repeated_diffusion_steps=1),
    dict(name="rds4", ablate_state="none", ablate_fov="none", repeated_diffusion_steps=4),
]

for setting in SMOKE_SETTINGS:
    run_eval(batches=SMOKE_BATCHES, phase="smoke", **setting)

smoke_df = load_summaries("smoke")
smoke_df.to_csv(OUT_ROOT / "smoke_summary.csv", index=False)
smoke_df

## Full Test-Split Ablation

Full eval is expensive. Turn `RUN_FULL` on only after smoke results look useful. Suggested minimum: `no_state`; suggested second setting: `shuffle_state`.

In [ ]:
RUN_FULL = False

FULL_SETTINGS = [
    dict(name="no_state_full", ablate_state="no_state", ablate_fov="none"),
    dict(name="shuffle_state_full", ablate_state="shuffle_state", ablate_fov="none"),
    # Uncomment if smoke shows a clear FOV effect and you still have GPU time.
    # dict(name="zero_fov_full", ablate_state="none", ablate_fov="zero_fov"),
]

if RUN_FULL:
    for setting in FULL_SETTINGS:
        run_eval(batches=FULL_BATCHES, phase="full", **setting)

full_df = load_summaries("full")
if not full_df.empty:
    full_df.to_csv(OUT_ROOT / "full_summary.csv", index=False)
full_df

In [ ]:
combined = pd.concat([load_summaries("smoke"), load_summaries("full")], ignore_index=True)
if not combined.empty:
    combined.to_csv(OUT_ROOT / "ablation_summary.csv", index=False)
    print(combined.to_string(index=False))
combined

In [ ]:
%%bash
set -e
zip -qr ../vitra_ablation_results.zip ablation_results
ls -lh ../vitra_ablation_results.zip